In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------ CONFIG ------------------
benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]
loads = [i for i in range(1, 10)]

input_dir = "/home/hsd/workspace/trafpy/examples/comparison_generator/final_data"
output_dir = "/home/hsd/workspace/ns3-load-balance/results_fix_asym"

save_dir = "/home/hsd/workspace/ns3-load-balance/results_fix_asym/analysis_results"
Path(save_dir).mkdir(exist_ok=True)

ip_pattern = r'^\d+\.\d+\.\d+\.\d+$'

summary_results = {}

# ------------------ FUNCTIONS ------------------
def clean(df):
    return df[
        df['Src'].str.match(ip_pattern, na=False) &
        df['Dest'].str.match(ip_pattern, na=False)
    ].copy()

def ip_to_node(ip):
    last = int(ip.split('.')[-1])
    last1 = int(ip.split('.')[-2])

    leafcount = 2
    leaf = 0
    if last1 == 2:
        leaf = 1

    return ((last // 2) - 1 + (leaf * leafcount))

def add_time(df):
    df['start_time'] = (
        df['TimeFirstTxPacket']
        .astype(str)
        .str.replace('+','', regex=False)
        .str.replace('ns','', regex=False)
        .astype(float) / 1e9
    )
    return df

# FlowMonitor columns
flow_columns = [
    "FlowID","Src","Dest","TimeFirstRxPacket","TimeFirstTxPacket",
    "TimeLastRxPacket","TimeLastTxPacket","FCT(s)","TxPackets",
    "RxPackets","LostPackets","LossRate","PDR","LossPercent",
    "TxBytes","RxBytes","Throughput(Kbps)","MeanDelay(ms)",
    "Jitter(ms)","HopCount"
]

# ------------------ MATCH FUNCTION ------------------
def match_df(output_df, input_df, label):
    matches = []

    for _, in_row in input_df.iterrows():
        cand = output_df[
            (output_df.sn == in_row.sn) &
            (output_df.dn == in_row.dn)
        ].copy()

        if len(cand) == 0:
            continue

        cand['time_diff'] = abs(cand['start_time'] - in_row.event_time)
        best = cand.loc[cand['time_diff'].idxmin()]

        combined = {
            'flow_id': in_row.flow_id,
            'sn': in_row.sn,
            'dn': in_row.dn,
            'input_time': in_row.event_time,
            'flow_size': in_row.flow_size,

            f'time_diff_{label}': best['time_diff'],

            # ✅ Separate packet counts
            f'TxPackets_{label}': best['TxPackets'],
            f'RxPackets_{label}': best['RxPackets'],

            # ✅ Derived bytes
            f'TxBytes_calc_{label}': best['TxPackets'] * 1400,
            f'RxBytes_calc_{label}': best['RxPackets'] * 1400,
        }

        # Add FlowMonitor columns with suffix
        for col in flow_columns:
            combined[f"{col}_{label}"] = best[col]

        matches.append(combined)

    return pd.DataFrame(matches)

# ------------------ MAIN LOOP ------------------
for benchmark in benchmarks:
    for load in loads:

        print(f"\n=== {benchmark} | Load 0.{load} ===")

        input_path = f"{input_dir}/{benchmark}_load_{load}.csv"
        conga_path = f"{output_dir}/{benchmark}_load_{load}_Conga.csv"
        ecmp_path = f"{output_dir}/{benchmark}_load_{load}_ECMP.csv"

        try:
            input_df = pd.read_csv(input_path)
            df1 = pd.read_csv(conga_path, nrows=12000)
            df2 = pd.read_csv(ecmp_path, nrows=12000)
        except Exception as e:
            print("Skipping:", e)
            continue

        # ------------------ CLEAN ------------------
        df1 = add_time(clean(df1))
        df2 = add_time(clean(df2))

        df1['sn'] = df1['Src'].apply(ip_to_node)
        df1['dn'] = df1['Dest'].apply(ip_to_node)

        df2['sn'] = df2['Src'].apply(ip_to_node)
        df2['dn'] = df2['Dest'].apply(ip_to_node)

        # ------------------ MATCH ------------------
        matched_conga = match_df(df1, input_df, "conga")
        matched_ecmp = match_df(df2, input_df, "ecmp")

        final_df = pd.merge(
            matched_conga,
            matched_ecmp,
            on=['flow_id', 'sn', 'dn', 'input_time', 'flow_size'],
            how='inner'
        )

        if len(final_df) == 0:
            print("No matches, skipping")
            continue

        # ------------------ DIFFS ------------------
        final_df['fct_diff'] = (
            final_df['FCT(s)_conga'] - final_df['FCT(s)_ecmp']
        )

        final_df['tx_diff'] = (
            final_df['TxPackets_conga'] - final_df['TxPackets_ecmp']
        )

        final_df['rx_diff'] = (
            final_df['RxPackets_conga'] - final_df['RxPackets_ecmp']
        )

        # ------------------ SAVE CSV ------------------
        csv_path = f"{save_dir}/{benchmark}_load_{load}_full.csv"
        final_df.to_csv(csv_path, index=False)

        # ------------------ STATS ------------------
        avg_conga = final_df['FCT(s)_conga'].mean()
        avg_ecmp = final_df['FCT(s)_ecmp'].mean()

        print("Flows:", len(final_df))
        print("Avg Conga:", avg_conga)
        print("Avg ECMP:", avg_ecmp)

        # ------------------ STORE SUMMARY ------------------
        if benchmark not in summary_results:
            summary_results[benchmark] = {
                'loads': [],
                'conga': [],
                'ecmp': []
            }

        summary_results[benchmark]['loads'].append(load / 10)
        summary_results[benchmark]['conga'].append(avg_conga)
        summary_results[benchmark]['ecmp'].append(avg_ecmp)

        # ------------------ PLOTS ------------------
        max_val = max(final_df['FCT(s)_conga'].max(),
                      final_df['FCT(s)_ecmp'].max())

        # Scatter
        plt.figure()
        plt.scatter(final_df['FCT(s)_conga'], final_df['FCT(s)_ecmp'], alpha=0.5)
        plt.plot([0, max_val], [0, max_val], linestyle='--')
        plt.xlabel("Conga FCT")
        plt.ylabel("ECMP FCT")
        plt.title(f"{benchmark} Load 0.{load}")
        plt.grid()
        plt.savefig(f"{save_dir}/{benchmark}_load_{load}_scatter.png")
        plt.close()

        # Log Scatter
        plt.figure()
        plt.scatter(final_df['FCT(s)_conga'], final_df['FCT(s)_ecmp'], alpha=0.5)
        plt.xscale('log')
        plt.yscale('log')
        plt.xlabel("Conga FCT (log)")
        plt.ylabel("ECMP FCT (log)")
        plt.title(f"{benchmark} Load 0.{load}")
        plt.grid()
        plt.savefig(f"{save_dir}/{benchmark}_load_{load}_log.png")
        plt.close()

        # Histogram
        plt.figure()
        plt.hist(final_df['fct_diff'], bins=30)
        plt.xlabel("FCT Diff (Conga - ECMP)")
        plt.ylabel("Count")
        plt.title(f"{benchmark} Load 0.{load}")
        plt.grid()
        plt.savefig(f"{save_dir}/{benchmark}_load_{load}_hist.png")
        plt.close()

# ------------------ FINAL LINE PLOTS ------------------
for benchmark in summary_results:

    loads = summary_results[benchmark]['loads']
    conga_avg = summary_results[benchmark]['conga']
    ecmp_avg = summary_results[benchmark]['ecmp']

    plt.figure()
    plt.plot(loads, conga_avg, marker='o', label='Conga')
    plt.plot(loads, ecmp_avg, marker='s', label='ECMP')

    plt.xlabel("Load")
    plt.ylabel("Average FCT")
    plt.title(f"{benchmark} Avg FCT vs Load")
    plt.legend()
    plt.grid()

    plt.savefig(f"{save_dir}/{benchmark}_avg_fct_vs_load.png")
    plt.close()

print("\n DONE: Full pipeline with rich data + plots completed!")


=== private_enterprise | Load 0.1 ===


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------
save_dir = Path("/home/hsd/workspace/ns3-load-balance/results_fix_asym/analysis_results/")

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]
loads = [i for i in range(1, 10)]

SMALL = 100 * 1024        # 100 KB
LARGE = 1 * 1024 * 1024   # 1 MB

# ------------------ ANALYSIS ------------------
for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []
    
    small_conga, small_ecmp = [], []
    large_conga, large_ecmp = [], []

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        if not csv_path.exists():
            print(f"Missing: {csv_path}")
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        # ------------------ SPLIT FLOWS ------------------
        small_df = df[df['flow_size'] < SMALL]
        large_df = df[df['flow_size'] > LARGE]

        # ------------------ SMALL FLOWS ------------------
        if not small_df.empty:
            small_conga.append(small_df['FCT(s)_conga'].mean())
            small_ecmp.append(small_df['FCT(s)_ecmp'].mean())
        else:
            small_conga.append(np.nan)
            small_ecmp.append(np.nan)

        # ------------------ LARGE FLOWS ------------------
        if not large_df.empty:
            large_conga.append(large_df['FCT(s)_conga'].mean())
            large_ecmp.append(large_df['FCT(s)_ecmp'].mean())
        else:
            large_conga.append(np.nan)
            large_ecmp.append(np.nan)

        load_vals.append(load / 10)

    # ================== PLOT: SMALL FLOWS ==================
    plt.figure()

    plt.plot(load_vals, small_conga, marker='o', label='Conga')
    plt.plot(load_vals, small_ecmp, marker='s', label='ECMP')

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")
    plt.title(f"{benchmark}: Small Flows (<100KB)")
    plt.legend()
    plt.grid()

    out_path = save_dir / f"graph/{benchmark}_SMALL_vs_load.png"
    plt.savefig(out_path)
    plt.close()

    print(f"Saved: {out_path}")

    # ================== PLOT: LARGE FLOWS ==================
    plt.figure()

    plt.plot(load_vals, large_conga, marker='o', label='Conga')
    plt.plot(load_vals, large_ecmp, marker='s', label='ECMP')

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")
    plt.title(f"{benchmark}: Large Flows (>1MB)")
    plt.legend()
    plt.grid()

    out_path = save_dir / f"graph/{benchmark}_LARGE_vs_load.png"
    plt.savefig(out_path)
    plt.close()

    print(f"Saved: {out_path}")

print("\n Done: Small vs Large flow comparison (post-processing)")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------
save_dir = Path("/home/hsd/workspace/ns3-load-balance/results_fix_asym/analysis_results/")

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]
loads = [i for i in range(1, 10)]

SMALL = 100 * 1024        # 100 KB
LARGE = 1 * 1024 * 1024   # 1 MB

# ------------------ MAIN ------------------
for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    small_conga, small_ecmp = [], []
    large_conga, large_ecmp = [], []
    load_vals = []

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"
        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)
        if len(df) == 0:
            continue

        # ------------------ SPLIT ------------------
        small_df = df[df['flow_size'] < SMALL]
        large_df = df[df['flow_size'] > LARGE]

        if len(small_df) > 0:
            small_conga.append(small_df['FCT(s)_conga'].mean())
            small_ecmp.append(small_df['FCT(s)_ecmp'].mean())
        else:
            small_conga.append(np.nan)
            small_ecmp.append(np.nan)

        if len(large_df) > 0:
            large_conga.append(large_df['FCT(s)_conga'].mean())
            large_ecmp.append(large_df['FCT(s)_ecmp'].mean())
        else:
            large_conga.append(np.nan)
            large_ecmp.append(np.nan)

        load_vals.append(load / 10)

        # ================== COLORED SCATTER ==================
        plt.figure(figsize=(6,6))

        # Masks
        small_mask = df['flow_size'] < SMALL
        large_mask = df['flow_size'] > LARGE
        mid_mask = ~(small_mask | large_mask)

        # Plot each category
        plt.scatter(
            df.loc[small_mask, 'FCT(s)_conga'],
            df.loc[small_mask, 'FCT(s)_ecmp'],
            alpha=0.5, label='Small (<100KB)'
        )

        plt.scatter(
            df.loc[mid_mask, 'FCT(s)_conga'],
            df.loc[mid_mask, 'FCT(s)_ecmp'],
            alpha=0.5, label='Medium'
        )

        plt.scatter(
            df.loc[large_mask, 'FCT(s)_conga'],
            df.loc[large_mask, 'FCT(s)_ecmp'],
            alpha=0.5, label='Large (>1MB)'
        )

        # Diagonal
        max_val = max(df['FCT(s)_conga'].max(), df['FCT(s)_ecmp'].max())
        plt.plot([0, max_val], [0, max_val], linestyle='--')

        plt.xlabel("Conga FCT")
        plt.ylabel("ECMP FCT")
        plt.title(f"{benchmark} Load 0.{load} (Size-colored)")
        plt.legend()
        plt.grid()

        # Save + Show
        plt.savefig(save_dir / f"graph/{benchmark}_load_{load}_colored_scatter.png")
        plt.show()
        plt.close()

    

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------
save_dir = Path("/home/hsd/workspace/ns3-load-balance/results_fix_asym/analysis_results/")
graph_dir = save_dir / "graph/fct/normalised"
graph_dir.mkdir(exist_ok=True)

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]
loads = [i for i in range(1, 10)]

SMALL = 100 * 1024        # 100 KB
LARGE = 1 * 1024 * 1024   # 1 MB

# =========================================================
# MAIN
# =========================================================
for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []
    
    small_rel = []
    large_rel = []

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"
        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)
        if df.empty:
            continue

        # avoid divide-by-zero
        df = df[df['FCT(s)_ecmp'] > 0]

        # ------------------ NORMALIZATION ------------------
        df['rel_fct'] = df['FCT(s)_conga'] / df['FCT(s)_ecmp']

        # ------------------ SPLIT ------------------
        small_df = df[df['flow_size'] < SMALL]
        large_df = df[df['flow_size'] > LARGE]

        # small flows
        if len(small_df) > 0:
            small_rel.append(small_df['rel_fct'].mean())
        else:
            small_rel.append(np.nan)

        # large flows
        if len(large_df) > 0:
            large_rel.append(large_df['rel_fct'].mean())
        else:
            large_rel.append(np.nan)

        load_vals.append(load / 10)

    # ================== SMALL FLOWS GRAPH ==================
    plt.figure(figsize=(8,6), dpi=120)

    plt.plot(load_vals, small_rel, marker='o')

    plt.axhline(y=1, linestyle='--')  # ECMP baseline

    plt.xlabel("Network Load")
    plt.ylabel("Relative FCT (Conga / ECMP)")
    plt.title(f"{benchmark}: Small Flows (<100KB)")
    plt.grid()

    # FIXED SCALE
    plt.ylim(0, 1.6)
    plt.yticks(np.arange(0, 1.6 + 0.2, 0.2))

    out_path = graph_dir / f"{benchmark}_SMALL_relative_vs_load.png"
    plt.savefig(out_path)
    plt.close()

    print(f"Saved: {out_path}")

    # ================== LARGE FLOWS GRAPH ==================
    plt.figure(figsize=(8,6), dpi=120)

    plt.plot(load_vals, large_rel, marker='o')

    plt.axhline(y=1, linestyle='--')  # ECMP baseline

    plt.xlabel("Network Load")
    plt.ylabel("Relative FCT (Conga / ECMP)")
    plt.title(f"{benchmark}: Large Flows (>1MB)")
    plt.grid()

    # FIXED SCALE
    plt.ylim(0, 1.2)
    plt.yticks(np.arange(0, 1.2 + 0.2, 0.2))

    out_path = graph_dir / f"{benchmark}_LARGE_relative_vs_load.png"
    plt.savefig(out_path)
    plt.close()

    print(f"Saved: {out_path}")

print("\n Done: 8 normalized graphs generated")